# RAG Pipeline — Walkthrough Notebook

This notebook walks you through the Historical RAG pipeline stage by stage. At each stage you see:

* **What goes in** (input data shape)
* **What the code does** (the actual function or formula)
* **What comes out** (output data shape)

External services (Azure AI Search, the LLM gateway) are stubbed with realistic mock data so the notebook runs offline. Companion reference docs:

* [docs/rag.md](rag.md) — full design reference
* [docs/ingestion.md](ingestion.md) — how the indexed documents were built

## Stages

1. Query preparation — clean & (optionally) condense the idea card
2. Semantic retrieval — value-stream catalog (Azure)
3. Historical retrieval — analog tickets (Azure or FAISS) → per-VS support
4. Merging — lanes, gates, sort, window-fill
5. Review-Pool LLM call — final selection
6. Optional post-processing — themes & stage prediction
7. Final response shape

## 0. Imports & sample input

We import the real pipeline pieces where they don't require external calls, and stub everything else.

In [ ]:
from __future__ import annotations

import json
import math
import sys
from pathlib import Path
from pprint import pprint

REPO = Path.cwd().resolve()
if (REPO / 'src').exists() and str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

# Real, side-effect-free helpers from the codebase:
from vs_app.modules.rag.query.views import (
    clean_ppt_text,
    normalize_for_search,
)
from vs_app.modules.rag.retrieval.semantic_retriever import _semantic_search_query
from vs_app.modules.rag.augmentation.candidate_merger import (
    GENERIC_OR_RISKY_STREAMS,
    assign_lane,
    historical_support_weight,
)
from vs_app.ingestion.summary.mapper import format_structured_summary_text

print('imports ok')

In [ ]:
# A small but realistic idea-card. In production this is extracted from a Jira
# attachment (pptx / pdf / docx) by `extract_idea_card_text`. We use a short
# one here so the condense LLM short-circuit branch fires (length <= 3500).

IDEA_CARD_TEXT = '''
Idea Card Title: CP 2025 National Network Product Tiering and Steerage (Coupe Health)

Description: Configure benefit tiering and member steerage logic for the new
Coupe Health national network product. Updates affect benefit design,
provider tier assignments, and the EOB messages members receive when they
see an out-of-tier provider. Claims should re-price against the new tier
structure starting on the effective date; member service must be able to
explain the tier and steerage to callers.

Stakeholders: members enrolled in Coupe Health plans, network operations,
claims operations, member service.
Systems: Facets, EPDB, member portal.
'''.strip()

print(f'length = {len(IDEA_CARD_TEXT)} chars')
print('first 200 chars:', IDEA_CARD_TEXT[:200], '...')

## 1. Query preparation

Two transformations turn the raw idea-card text into the retrieval query:

| Step | Function | Purpose |
| ---- | -------- | ------- |
| 1.1  | `clean_ppt_text` | Strip OCR junk, bullets, double letters, table noise |
| 1.2  | `condense_idea_card` | If cleaned length > 3500, LLM-summarize into structured shape. Otherwise pass through. |
| 1.3  | `normalize_for_search` (semantic only) | Lowercase, strip Azure-disallowed chars |
| 1.4  | `_semantic_search_query` (semantic only) | Token dedupe, cap to 90 unique tokens |

In [ ]:
# 1.1 Clean
cleaned_query = clean_ppt_text(IDEA_CARD_TEXT)
print('cleaned length:', len(cleaned_query))
print('--- cleaned ---')
print(cleaned_query)

In [ ]:
# 1.2 Condense decision
#
# Real code:
#     def condense_idea_card_with_metadata(raw_text, max_chars=3500):
#         cleaned = clean_opt_text(raw_text)
#         if len(cleaned) <= max_chars:
#             return cleaned[:max_chars], {}   # SHORT-CIRCUIT
#         # else: call LLM with retrieval_summary.yaml prompt

MAX_CHARS = 3500
needs_condense = len(cleaned_query) > MAX_CHARS
print(f'cleaned length = {len(cleaned_query)}, threshold = {MAX_CHARS}, condense LLM call? {needs_condense}')

if not needs_condense:
    query_for_prompt = cleaned_query[:MAX_CHARS]
else:
    # Demonstrate what the LLM *would* be asked to produce. In production this
    # JSON is parsed and flattened by `format_structured_summary_text(...)`.
    simulated_llm_output = {
        'summary_text': 'Configure benefit tiering and member steerage for the Coupe Health national network product, affecting benefit design, provider tier assignment, claim re-pricing, and member-facing EOB messaging.',
        'business_problem': 'Members and operations lack a configured tiering and steerage model for the new Coupe Health product.',
        'business_capability': 'Configure provider tiers, steer members to in-tier providers, and re-price claims under the new tier structure.',
        'stakeholders': ['members', 'network operations', 'claims operations', 'member service'],
        'systems_and_products': ['Facets', 'EPDB', 'member portal', 'Coupe Health'],
        'key_terms': ['benefit tiering', 'steerage', 'EOB', 'tier assignment', 're-pricing'],
    }
    query_for_prompt = format_structured_summary_text(simulated_llm_output, max_chars=MAX_CHARS)

retrieval_query = query_for_prompt or cleaned_query
print('--- query_for_prompt ---')
print(query_for_prompt)

In [ ]:
# 1.3 + 1.4 Semantic-side preprocessing
normalized = normalize_for_search(retrieval_query)
semantic_query = _semantic_search_query(retrieval_query)

print('--- normalized (first 200) ---')
print(normalized[:200])
print()
print('--- semantic_query (deduped, <= 90 unique tokens, no 1-char tokens) ---')
print(semantic_query)
print()
print(f'token count = {len(semantic_query.split())}')

## 2. Semantic retrieval — value-stream catalog

Real call (against index `value-streams`):

```python
client.search_hybrid(
    _semantic_search_query(query),
    top_k=60,
    use_semantic_rerank=True,
    filter_expression="node_type eq 'ValueStream'",
    search_fields=['entity_name', 'content'],
)
```

Three matchers fire **in one call**:

* **BM25** over `entity_name` and `content`
* **Vector kNN** over `content_vector`
* **Semantic reranker** re-orders the RRF-fused list

The picked score on each hit is `@search.reranker_score` if present, else `@search.score`.

Below is a stub of what the call returns and what we keep.

In [ ]:
# Mock Azure response for the VS catalog. Real shape — these are the only
# fields the retriever reads downstream.
mock_semantic_hits = [
    {
        'entity_id': 'vs-001',
        'entity_name': 'Establish Product Offering',
        'content': 'Design, configure, and approve new health plan products including benefit design, network configuration, and pricing.',
        '@search.score': 14.2,
        '@search.reranker_score': 1.78,
    },
    {
        'entity_id': 'vs-002',
        'entity_name': 'Adjudicate Claim',
        'content': 'Receive a claim and determine the payable amount based on benefits, eligibility, contracts, and provider tier.',
        '@search.score': 12.1,
        '@search.reranker_score': 1.55,
    },
    {
        'entity_id': 'vs-003',
        'entity_name': 'Manage Member Care',
        'content': 'Support members through care interactions including service inquiries, communications, and steerage.',
        '@search.score': 9.4,
        '@search.reranker_score': 1.42,
    },
    {
        'entity_id': 'vs-004',
        'entity_name': 'Configure Provider Network',
        'content': 'Build and maintain provider networks, tiers, and contractual arrangements.',
        '@search.score': 11.6,
        '@search.reranker_score': 1.50,
    },
    {
        'entity_id': 'vs-005',
        'entity_name': 'Discover Business Insights',
        'content': 'Analytics and reporting across the enterprise.',
        '@search.score': 6.1,
        '@search.reranker_score': 1.10,
    },
]

# Mimic `retrieve_semantic_candidates`: prefer reranker score, dedupe, sort.
semantic_candidates = []
for row in mock_semantic_hits:
    score = row['@search.reranker_score'] if row.get('@search.reranker_score') is not None else row['@search.score']
    semantic_candidates.append({
        'entity_id': row['entity_id'],
        'entity_name': row['entity_name'],
        'description': row['content'],
        'semantic_score': round(float(score), 4),
        'from_semantic': True,
        'from_historical': False,
    })
semantic_candidates.sort(key=lambda r: -r['semantic_score'])

print(f'{len(semantic_candidates)} semantic candidates (sorted by score):')
for row in semantic_candidates:
    print(f"  {row['semantic_score']:>6.4f}  {row['entity_name']}")

## 3. Historical retrieval — analog tickets

Real call against `idp_idmt_data`:

```python
client.search_hybrid(
    query,
    vector_field='content_vector',
    search_fields=['content', 'summary_text', 'business_problem', 'business_capability'],
    use_semantic_rerank=False,        # multi-field BM25 + vector is enough
)
```

Each hit is a past ticket with `direct_vs_names` / `implied_vs_names` already attached at ingestion. The retriever then **groups by value-stream name** and aggregates per-VS support.

In [ ]:
# Mock historical ticket hits. Real shape returned by `search_historical_summaries`.
mock_historical_hits = [
    {
        'ticket_id': 'IDMT-4491',
        'best_score': 0.82,
        'title': 'BCBS NJ Tier 1 Steerage',
        'summary_preview': 'Configure tier 1 steerage and re-price claims under new product.',
        'direct_vs_names': ['Establish Product Offering', 'Configure Provider Network'],
        'implied_vs_names': ['Adjudicate Claim'],
        'label_source': 'jira_issuelinks',
    },
    {
        'ticket_id': 'IDMT-5012',
        'best_score': 0.76,
        'title': 'Empire EOB Tier Messaging',
        'summary_preview': 'Update member EOB messages for new tier definitions.',
        'direct_vs_names': ['Manage Member Care'],
        'implied_vs_names': ['Establish Product Offering'],
        'label_source': 'jira_issuelinks',
    },
    {
        'ticket_id': 'IDMT-6188',
        'best_score': 0.71,
        'title': 'Highmark Steerage Logic',
        'summary_preview': 'Member steerage rule updates and benefit tier re-pricing.',
        'direct_vs_names': ['Establish Product Offering'],
        'implied_vs_names': ['Adjudicate Claim', 'Manage Member Care'],
        'label_source': 'jira_issuelinks',
    },
]

for h in mock_historical_hits:
    print(f"{h['ticket_id']:10} score={h['best_score']:.3f}  direct={h['direct_vs_names']}  implied={h['implied_vs_names']}")

In [ ]:
# Now reproduce `_build_support_from_faiss_hits` end-to-end so you can SEE the math.
#
# Per-ticket weight = 1 / n_streams_on_ticket. Streams already direct stay direct;
# names appearing in both arrays are kept only in the direct set.

support = {}  # entity_name -> aggregate

for hit in mock_historical_hits:
    direct_set = {n for n in hit['direct_vs_names']}
    implied = [n for n in hit['implied_vs_names'] if n not in direct_set]
    stream_names = list(hit['direct_vs_names']) + implied
    n = max(len(stream_names), 1)
    per_ticket_weight = 1.0 / n
    score = hit['best_score']

    for name in stream_names:
        is_direct = name in direct_set
        entry = support.setdefault(name, {
            'entity_name': name,
            'support_count': 0,
            'direct_count': 0,
            'implied_count': 0,
            'weighted_support_count': 0.0,
            'weighted_direct_count': 0.0,
            'weighted_implied_count': 0.0,
            'best_support_score': 0.0,
            'total_score': 0.0,
            'supporting_ticket_ids': [],
        })
        entry['support_count'] += 1
        entry['weighted_support_count'] += per_ticket_weight
        entry['best_support_score'] = max(entry['best_support_score'], score)
        entry['total_score'] += score
        if is_direct:
            entry['direct_count'] += 1
            entry['weighted_direct_count'] += per_ticket_weight
        else:
            entry['implied_count'] += 1
            entry['weighted_implied_count'] += per_ticket_weight
        if hit['ticket_id'] not in entry['supporting_ticket_ids']:
            entry['supporting_ticket_ids'].append(hit['ticket_id'])

# Finalize: avg_support_score, rounding, sort
historical_value_stream_support = []
for entry in support.values():
    cnt = max(entry['support_count'], 1)
    entry['avg_support_score'] = round(entry.pop('total_score') / cnt, 4)
    entry['best_support_score'] = round(entry['best_support_score'], 4)
    entry['weighted_support_count'] = round(entry['weighted_support_count'], 4)
    entry['weighted_direct_count'] = round(entry['weighted_direct_count'], 4)
    entry['weighted_implied_count'] = round(entry['weighted_implied_count'], 4)
    historical_value_stream_support.append(entry)

historical_value_stream_support.sort(key=lambda r: (
    -r['weighted_support_count'],
    -r['weighted_direct_count'],
    -r['best_support_score'],
))

print('Per-value-stream aggregates (sorted):\n')
for row in historical_value_stream_support:
    print(
        f"  {row['entity_name']:34}  hits={row['support_count']}  "
        f"direct={row['direct_count']}  implied={row['implied_count']}  "
        f"weighted={row['weighted_support_count']:.4f}  best={row['best_support_score']:.3f}"
    )

## 4. Merging into the LLM candidate window

Three steps:

1. **Union by normalized name** — combine semantic + historical rows.
2. **Assign a lane** based purely on evidence presence (not score).
3. **Gate, sort, and window-fill** in fixed priority order.

In [ ]:
# 4.1 Union
def norm(name):
    return ' '.join((name or '').strip().lower().split())

merged = {}
for row in semantic_candidates:
    merged[norm(row['entity_name'])] = {
        **row,
        'support_count': 0,
        'direct_count': 0,
        'implied_count': 0,
        'best_support_score': 0.0,
        'avg_support_score': 0.0,
        'weighted_support_count': 0.0,
        'weighted_direct_count': 0.0,
        'weighted_implied_count': 0.0,
        'supporting_ticket_ids': [],
    }

for hist in historical_value_stream_support:
    key = norm(hist['entity_name'])
    if key in merged:
        merged[key].update({
            'from_historical': True,
            'support_count': hist['support_count'],
            'direct_count': hist['direct_count'],
            'implied_count': hist['implied_count'],
            'best_support_score': hist['best_support_score'],
            'avg_support_score': hist['avg_support_score'],
            'weighted_support_count': hist['weighted_support_count'],
            'weighted_direct_count': hist['weighted_direct_count'],
            'weighted_implied_count': hist['weighted_implied_count'],
            'supporting_ticket_ids': hist['supporting_ticket_ids'],
        })
    else:
        merged[key] = {
            'entity_id': '',
            'entity_name': hist['entity_name'],
            'description': '',
            'semantic_score': 0.0,
            'from_semantic': False,
            'from_historical': True,
            **{k: hist[k] for k in (
                'support_count','direct_count','implied_count',
                'best_support_score','avg_support_score',
                'weighted_support_count','weighted_direct_count','weighted_implied_count',
                'supporting_ticket_ids',
            )},
        }

# 4.2 Assign lane using the real helper
for row in merged.values():
    row['lane'] = assign_lane(row)

print('After union + lane assignment:\n')
print(f"  {'lane':28} {'name':36} sem    hits  best   weighted")
for row in merged.values():
    print(
        f"  {row['lane']:28} {row['entity_name']:36} "
        f"{row['semantic_score']:5.3f}  {row['support_count']:>3}   "
        f"{row['best_support_score']:5.3f}  {row['weighted_support_count']:6.3f}"
    )

In [ ]:
# 4.3 Gates - which candidates are eligible for the LLM window?

def historical_only_passes(row):
    """At least one of: hits>=2, direct>=1, best>=0.65, weighted>=0.6."""
    return (
        row['support_count'] >= 2
        or row['direct_count'] >= 1
        or row['best_support_score'] >= 0.65
        or row['weighted_support_count'] >= 0.6
    )

def semantic_only_passes(row):
    """semantic_score >= 1.20 (or >= 1.35 if generic)."""
    floor = 1.35 if norm(row['entity_name']) in GENERIC_OR_RISKY_STREAMS else 1.20
    return row['semantic_score'] >= floor

for row in merged.values():
    if row['lane'] == 'semantic_plus_historical':
        gate = 'no-gate (strongest lane)'
    elif row['lane'] == 'historical_only':
        gate = f"pass={historical_only_passes(row)}"
    elif row['lane'] == 'semantic_only':
        gate = f"pass={semantic_only_passes(row)}"
    else:
        gate = 'n/a'
    is_generic = norm(row['entity_name']) in GENERIC_OR_RISKY_STREAMS
    print(f"  {row['lane']:28} {row['entity_name']:36} generic={is_generic!s:5} -> {gate}")

In [ ]:
# 4.4 Blended sort for the merged (semantic + historical) lane
#
#   historical_boost = min(1.0, hits/10.0) * 0.20 + best_support * 0.15
#   blended = semantic_score + historical_boost
#   if generic and hits < 3:
#       blended -= 0.20

def blended_score(row):
    hits = row['support_count']
    boost = min(1.0, hits / 10.0) * 0.20 + row['best_support_score'] * 0.15
    blended = row['semantic_score'] + boost
    if norm(row['entity_name']) in GENERIC_OR_RISKY_STREAMS and hits < 3:
        blended -= 0.20
    return round(blended, 4)

merged_lane = [r for r in merged.values() if r['lane'] == 'semantic_plus_historical']
merged_lane.sort(key=lambda r: -blended_score(r))

print('semantic_plus_historical lane after blended sort:\n')
for r in merged_lane:
    print(f"  blended={blended_score(r):.4f}  sem={r['semantic_score']:.3f}  hits={r['support_count']}  best={r['best_support_score']:.3f}  -> {r['entity_name']}")

In [ ]:
# 4.5 Fallback weight quantization (only used when a backend doesn't supply weighted_support)

for s in (0.85, 0.78, 0.72, 0.68, 0.61, 0.55):
    print(f'  similarity {s:.2f}  ->  fallback weight = {historical_support_weight(s):.1f}')

In [ ]:
# 4.6 Window fill (strict priority)
#
#   1. merged (semantic_plus_historical) up to max_semantic_plus_historical
#   2. historical_only that passes gate, up to max_historical_only
#   3. semantic_only that passes gate, up to max_semantic_only
#   stop at llm_candidate_window

llm_candidate_window = 35   # default for requested=12
max_merged           = llm_candidate_window
max_historical_only  = 5
max_semantic_only    = 4

hist_only_lane = sorted(
    [r for r in merged.values() if r['lane'] == 'historical_only' and historical_only_passes(r)],
    key=lambda r: (-r['best_support_score'], -r['weighted_support_count']),
)
sem_only_lane = sorted(
    [r for r in merged.values() if r['lane'] == 'semantic_only' and semantic_only_passes(r)],
    key=lambda r: -r['semantic_score'],
)

llm_candidates = []
llm_candidates += merged_lane[:max_merged]
remaining = llm_candidate_window - len(llm_candidates)
llm_candidates += hist_only_lane[: min(max_historical_only, remaining)]
remaining = llm_candidate_window - len(llm_candidates)
llm_candidates += sem_only_lane[: min(max_semantic_only, remaining)]

print(f'Filled LLM window ({len(llm_candidates)} / {llm_candidate_window}):\n')
for idx, r in enumerate(llm_candidates, start=1):
    print(f'  {idx}. [{r["lane"]}]  {r["entity_name"]}')

print()
print('Counts by lane in the window:')
from collections import Counter
print('  ', dict(Counter(r['lane'] for r in llm_candidates)))

## 5. Review-Pool LLM call

The candidate window is rendered as a numbered list of blocks (one per candidate), then handed to the LLM along with the **system prompt** (defines the task) and the **user prompt** (carries the idea card + candidate blocks).

Real call:

```python
result = GenerationService().generate_structured(
    query=prompt,
    output_schema=ReviewPoolPickResult,            # picks: list[{entity_id, confidence, reason}]
    system_prompt=system_prompt,
    reasoning_effort='low',
)
```

In [ ]:
# 5.1 Build one candidate block (this is what the LLM sees per candidate)
def format_candidate_block(idx, row, desc_chars=100, analog_chars=80):
    lines = [
        f"{idx}. {row['entity_name']}",
        f"Entity ID: {row['entity_id']}",
        f"Lane: {row['lane']}",
    ]
    desc = ' '.join((row.get('description') or '').split())
    if desc:
        lines.append(f"Description: {desc[:desc_chars]}")
    if row.get('from_semantic'):
        lines.append(f"Semantic score: {row['semantic_score']:.4f}")
    if row.get('from_historical'):
        lines.append(
            f"Historical: {row['support_count']} tickets "
            f"({row['direct_count']} direct, {row['implied_count']} implied), "
            f"best {row['best_support_score']:.3f}, avg {row['avg_support_score']:.3f}, "
            f"weighted {row['weighted_support_count']:.3f}"
        )
        if row.get('supporting_ticket_ids'):
            lines.append('Supporting tickets: ' + ', '.join(row['supporting_ticket_ids'][:2]))
    return '\n'.join(lines)

candidate_blocks = '\n\n'.join(
    format_candidate_block(i, r) for i, r in enumerate(llm_candidates, start=1)
)
print(candidate_blocks)

In [ ]:
# 5.2 Mock LLM response (what `generate_structured` would return).
# The model returns ONLY entity_id, confidence, reason. Names are filled
# downstream from the candidate list to prevent invented names.

mock_llm_picks = [
    {
        'entity_id': 'vs-001',
        'confidence': 0.92,
        'reason': 'Idea card directly configures benefit design and product setup for Coupe Health.',
    },
    {
        'entity_id': 'vs-004',
        'confidence': 0.80,
        'reason': 'Provider tier assignment is core to the steerage change.',
    },
    {
        'entity_id': 'vs-002',
        'confidence': 0.62,
        'reason': 'Claims must re-price against the new tier structure starting on the effective date.',
    },
    {
        'entity_id': 'vs-003',
        'confidence': 0.55,
        'reason': 'Member service must explain tier and steerage to callers.',
    },
]

# 5.3 Post-process: drop unknown ids, dedupe, build final rows
candidates_by_id = {r['entity_id'].lower(): r for r in llm_candidates if r['entity_id']}
selected = []
seen = set()
for pick in mock_llm_picks:
    pid = pick['entity_id'].lower()
    if not pid or pid in seen:
        continue
    candidate = candidates_by_id.get(pid)
    if candidate is None:
        print(f"  WARN dropping unknown entity_id '{pid}'")
        continue
    seen.add(pid)
    selected.append({
        'entity_id': candidate['entity_id'],
        'entity_name': candidate['entity_name'],
        'confidence': max(0.0, min(1.0, float(pick['confidence']))),
        'reason': pick['reason'],
        'selection_source': 'llm_pick',
        'supporting_ticket_ids': candidate.get('supporting_ticket_ids', [])[:5],
    })

print(f'\n{len(selected)} selected value streams:\n')
for s in selected:
    print(f"  conf={s['confidence']:.2f}  {s['entity_name']}")
    print(f"            reason: {s['reason']}")

In [ ]:
# 5.4 Safe backfill — only fires when LLM picked very few. With 4 picks here,
# we are below min_target=8 so backfill kicks in (from semantic_plus_historical
# candidates that meet the safer thresholds: sem>=1.05 OR hits>=3).

MIN_TARGET = min(12, 8)   # min(requested, 8)
if len(selected) < MIN_TARGET:
    already_ids = {s['entity_id'].lower() for s in selected}
    for cand in merged_lane:
        if len(selected) >= MIN_TARGET:
            break
        if cand['entity_id'].lower() in already_ids:
            continue
        if cand['lane'] != 'semantic_plus_historical':
            continue
        if cand['semantic_score'] < 1.05 and cand['support_count'] < 3:
            continue
        selected.append({
            'entity_id': cand['entity_id'],
            'entity_name': cand['entity_name'],
            'confidence': 0.35,
            'reason': 'Added as a low-confidence review candidate based on combined semantic and historical evidence.',
            'selection_source': 'safe_backfill',
            'supporting_ticket_ids': cand.get('supporting_ticket_ids', [])[:5],
        })
        already_ids.add(cand['entity_id'].lower())

print(f'After safe-backfill: {len(selected)} selected.\n')
for s in selected:
    print(f"  conf={s['confidence']:.2f}  [{s['selection_source']:13}] {s['entity_name']}")

## 6. Final response shape

The pipeline returns a single dict that the FastAPI route turns into a `ValueStreamRagResponse`. Key fields:

In [ ]:
final_response = {
    'selected_value_streams': selected,
    'llm_selected_value_streams': [s for s in selected if s['selection_source'] == 'llm_pick'],
    'auto_selected_value_streams': [],     # always [] in current pipeline
    'semantic_candidate_value_streams': semantic_candidates,
    'historical_candidate_value_streams': historical_value_stream_support,
    'merged_candidate_value_streams': list(merged.values()),
    'historical_ticket_hits': mock_historical_hits,
    'llm_candidates': llm_candidates,
    'query_preparation': {
        'cleaned_query': cleaned_query,
        'query_for_prompt': query_for_prompt,
    },
    'rag_runtime_config': {
        'final_output_count': 12,
        'semantic_fetch_k': 60,
        'historical_ticket_fetch_k': 60,
        'llm_candidate_window': llm_candidate_window,
    },
    'candidate_window_counts': dict(Counter(r['lane'] for r in llm_candidates)),
    'historical_source': 'summary_azure_ai_search',   # or 'summary_faiss', or 'none'
    'historical_excluded_ticket_ids': [],
}

print(json.dumps({
    'selected_value_streams': [
        {'entity_name': s['entity_name'], 'confidence': s['confidence'], 'selection_source': s['selection_source']}
        for s in final_response['selected_value_streams']
    ],
    'candidate_window_counts': final_response['candidate_window_counts'],
    'historical_source': final_response['historical_source'],
}, indent=2))

## 7. Recap — what each stage produced

| Stage | Function | Output size |
| ----- | -------- | ----------- |
| 1. Clean | `clean_ppt_text` | one string |
| 1. Condense | `condense_idea_card` (no-op below 3500 chars) | one string |
| 2. Semantic | `retrieve_semantic_candidates` | ≤ 60 catalog rows |
| 3. Historical (hits) | `retrieve_historical_support` | ≤ 60 ticket hits |
| 3. Historical (support) | `_build_support_from_faiss_hits` | per-VS aggregates |
| 4. Merge | `merge_candidate_sources` | merged candidates + LLM window |
| 5. LLM | `generate_review_pool_value_streams` | picks (entity_id + confidence + reason) |
| 5. Backfill | `_safe_backfill_review_pool` | up to min(requested, 8) |
| 6. Themes / stages | `build_theme_payloads`, `predict_stages` | optional enrichment |

### Things to try locally

* Change the idea card to something **longer than 3500 chars** to make the condense LLM branch fire (see §1).
* Toggle one of the streams to be in `GENERIC_OR_RISKY_STREAMS` and watch the blended-sort penalty change the ordering (see §4.4).
* Drop the historical hits for `Establish Product Offering` so it becomes `semantic_only` and watch the lane gate (sem ≥ 1.20) decide whether it survives.
* Lower `mock_llm_picks` to just one entry and re-run §5.4 — safe-backfill should top up from the strongest merged candidates only.